## Creating Source Data

In [2]:
# import nessessary libraries
import pandas as pd
import sqlite3
import json

### Create and save the Customer Dataset


In [3]:
customers = pd.DataFrame({
    "CustomerID": [
        "C001", "C002", "C003", "C004",
        "C004", "C005", "C006", "C007",
        "C008", "C009", "C010"
    ],

    "CustomerName": [
        "Amina", "Brian", "Carol", "David",
        "David", "Esther", "Faith", "George",
        "Halima", "Ian", "Jane"
    ],

    "Segment": [
        "Retail", "Corporate", "Retail", "SME",
        "SME", "Corporate", "SME", "Retail",
        "Corporate", "Retail", "SME"
    ],

    "RegionCode": [
        "NRB", "MSA", "NRB", "KSM",
        "KSM", "NKR", "NRB", "MSA",
        "KSM", "NKR", "NRB"
    ]
})

customers

# Saving the DataFrame to a CSV file
customers.to_csv('customers.csv', index=False)

### Create and Save Orders Dataset


In [4]:
orders = pd.DataFrame({
    "OrderID": [
        "O001", "O002", "O003", "O004", "O005",
        "O006", "O007", "O008", "O009", "O010",
        "O011", "O012", "O013", "O014", "O015"
    ],

    "CustomerID": [
        "C001", "C002", "C001", "C003", "C004",
        "C005", "C006", "C002", "C007", "C008",
        "C009", "C010", "C006", "C003", "C999"
    ],

    "OrderAmount": [
        4500, 12500, 3200, 6800, 9100,
        15000, 7600, 8300, 2500, 11000,
        5400, 6200, 4300, 7200, 9900
    ],

    "ProductCategory": [
        "Electronics", "Office", "Home", "Electronics", "Office",
        "Electronics", "Home", "Office", "Home", "Electronics",
        "Office", "Home", "Electronics", "Office", "Electronics"
    ]
})

orders

# Saving the DataFrame to a CSV file
orders.to_csv('orders.csv', index=False)

### Create the JSON region lookup

In [5]:
regions = [
    {"RegionCode": "NRB", "RegionName": "Nairobi"},
    {"RegionCode": "MSA", "RegionName": "Mombasa"},
    {"RegionCode": "KSM", "RegionName": "Kisumu"},
    {"RegionCode": "NKR", "RegionName": "Nakuru"}
]

with open("regions.json", "w") as file:
    json.dump(regions, file, indent=4)

## Inspect the sources


In [7]:
# Reload the data from the CSV files and JSON file
customers = pd.read_csv('customers.csv')
orders = pd.read_csv('orders.csv')
# view the data
print(customers.head())
print(orders.head())

# check the shape of the dataframes
print("Customers DataFrame shape:", customers.shape)
print("Orders DataFrame shape:", orders.shape)

  CustomerID CustomerName    Segment RegionCode
0       C001        Amina     Retail        NRB
1       C002        Brian  Corporate        MSA
2       C003        Carol     Retail        NRB
3       C004        David        SME        KSM
4       C004        David        SME        KSM
  OrderID CustomerID  OrderAmount ProductCategory
0    O001       C001         4500     Electronics
1    O002       C002        12500          Office
2    O003       C001         3200            Home
3    O004       C003         6800     Electronics
4    O005       C004         9100          Office
Customers DataFrame shape: (11, 4)
Orders DataFrame shape: (15, 4)


### Question1
In the customers.csv, one row represent one customer record
### Question2
In the orders.csv, one row represent on transaction or order.
### Question 3
No. The customer dataset is at customer-record grain, while the orders dataset is at order/transaction grain. One customer can have multiple orders.

## Detect Key Problems

In [11]:
# check for duplicates in the customers DataFrame
print(f"The number of duplicates in customers is: {customers['CustomerID'].duplicated().sum()}")
customers[
    customers["CustomerID"].duplicated(keep=False)
]

The number of duplicates in customers is: 1


,CustomerID,CustomerName,Segment,RegionCode
3,C004,David,SME,KSM
4,C004,David,SME,KSM


### Question4 
Because CustomerID is supposed to identify a single customer record. If the customer table contains duplicate keys, a JOIN can match one order to multiple customer rows, multiplying the order rows and potentially inflating sales totals.

In [13]:
# find unmatched records in the orders
print(f"Unmatched CustomerIDs in orders: {set(orders['CustomerID']) - set(customers['CustomerID'])}")

Unmatched CustomerIDs in orders: {'C999'}


### Question 5
C999 appears in orders but not in customers

### Question 6
 The missing customer record could be representing a legitimate transaction, a data-entry error, a missing source record, or another data-quality problem. Silently deleting it would remove a real order from the analysis.

 

## Create SQL Database


In [16]:
# connect to SQLite database 
conn = sqlite3.connect("retail_lab.db")

# Put the data into the database
customers.to_sql(
    "customers",
    conn,
    if_exists="replace",
    index=False
)
orders.to_sql(
    "orders",
    conn,
    if_exists="replace",
    index=False
)

# test the connection by querying the database
query = """
SELECT *
FROM customers
LIMIT 5;
"""

pd.read_sql_query(query, conn)

,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C002,Brian,Corporate,MSA
2,C003,Carol,Retail,NRB
3,C004,David,SME,KSM
4,C004,David,SME,KSM


## Select and Where


In [ ]:
# test the connection by querying the database
query = """
SELECT
    OrderID,
    CustomerID,
    OrderAmount
FROM orders;
"""

pd.read_sql_query(query, conn)

,OrderID,CustomerID,OrderAmount
0,O001,C001,4500
1,O002,C002,12500
2,O003,C001,3200
3,O004,C003,6800
4,O005,C004,9100
5,O006,C005,15000
6,O007,C006,7600
7,O008,C002,8300
8,O009,C007,2500
9,O010,C008,11000


In [18]:
# Customers who are only in the retail segment
query = """
SELECT *
FROM customers
WHERE Segment = 'Retail';
"""

pd.read_sql_query(query, conn)

,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C003,Carol,Retail,NRB
2,C007,George,Retail,MSA
3,C009,Ian,Retail,NKR


In [19]:
# customers who made orders greater than ksh 8000.00 
query = """
SELECT *
FROM orders
WHERE OrderAmount > 8000;
"""

pd.read_sql_query(query, conn)

,OrderID,CustomerID,OrderAmount,ProductCategory
0,O002,C002,12500,Office
1,O005,C004,9100,Office
2,O006,C005,15000,Electronics
3,O008,C002,8300,Office
4,O010,C008,11000,Electronics
5,O015,C999,9900,Electronics


In [20]:
# Check for electronics orders only
query = """
SELECT *
FROM orders
WHERE ProductCategory = 'Electronics';
"""

pd.read_sql_query(query, conn)

,OrderID,CustomerID,OrderAmount,ProductCategory
0,O001,C001,4500,Electronics
1,O004,C003,6800,Electronics
2,O006,C005,15000,Electronics
3,O010,C008,11000,Electronics
4,O013,C006,4300,Electronics
5,O015,C999,9900,Electronics


In [21]:
# check for electronics orders greater than ksh 8000.00
query = """
SELECT *
FROM orders
WHERE ProductCategory = 'Electronics'
AND OrderAmount > 8000;
"""

pd.read_sql_query(query, conn)

,OrderID,CustomerID,OrderAmount,ProductCategory
0,O006,C005,15000,Electronics
1,O010,C008,11000,Electronics
2,O015,C999,9900,Electronics


## Groupby and aggregate Functions
